### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

sys.path.insert(0, str(cfg.paths.project_root))

paths:
  project_root: /home/p84400019/projects/consciousness-llms/IT-LLMs/
  model_path: ${model.company}/${model.model_family}/${model.model_size}/${model.it}/
  data_dir: ${paths.project_root}data/${paths.model_path}
  data_activations_dir: ${paths.data_dir}activations/
  data_activations_file: ${paths.data_activations_dir}multi_prompt_activations.pkl
  data_phyid_dir: ${paths.data_dir}phyid/
  data_phyid_file: ${paths.data_phyid_dir}multi_prompt_phyid.pkl
  plot_dir: ${paths.project_root}plots/${paths.model_path}
  plot_activations_dir: ${paths.plot_dir}activations/
  plot_time_series_dir: ${paths.plot_activations_dir}time_series/
  plot_phyid_dir: ${paths.plot_dir}phyid/
model:
  shortcode: D2-16-A2
  hf_name: deepseek-ai/DeepSeek-V2-Lite
  company: deepseek
  model_family: deepseek-v2
  model_size: 16B-A2B
  it: base
  plot_name: DeepSeek V2 Lite
  color: '#8e9e00'
  apply_chat_template: 'no'
generation:
  max_new_tokens: 128
  default_prompts:
  - Imagine a future where humans h

### Loading the model

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

load_model = True

model_name = cfg.model.hf_name
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True
)
if load_model:
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map='auto', 
        attn_implementation='eager',  
        trust_remote_code=True
    )
    model.generation_config = GenerationConfig.from_pretrained(model_name)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
    model.eval()

    text = "The goal of life is to"
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs.to(model.device), max_new_tokens=10)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(result)

/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.57s/it]
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MoE Inference] new_x.shape=torch.Size([42, 2048]), topk_ids.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_before_mul.shape=torch.Size([7, 6, 2048]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_after_mul.shape=torch.Size([7, 6, 2048])
[MoE Inference] summed_out.shape=torch.Size([7, 2048])
[MoE Inference] y.shape=torch.Size([7, 2048]), topk_idx.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] y.shape=torch.Size([1, 7, 2048])
[MoE Inference] new_x.shape=torch.Size([42, 2048]), topk_ids.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_before_mul.shape=torch.Size([7, 6, 2048]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_after_mul.shape=torch.Size([7, 6, 2048])
[MoE Inference] summed_out.shape=torch.Size([7, 2048])
[MoE Inference] y.shape=torch.Size([7, 2048]), topk_idx.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] y.shape=torch.Siz

### Record activations, save them, and verify them

In [ ]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations
from IPython.core.debugger import Pdb

load_from_disk = False
data_activations_file = cfg.paths.data_activations_file
max_new_tokens=cfg.generation.max_new_tokens
prompts = cfg.generation.default_prompts

if not load_from_disk:
    recorder = ActivationRecorder(model, tokenizer)
    activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens)
    activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)
    activations.save(data_activations_file)

# Load the activations from disk.
loaded_activations = MultiPromptActivations.load(data_activations_file)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
loaded_activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = loaded_activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_idx, head_acts in attn.heads.items():
    print(f"Head {head_idx} activations:")
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)

DeepseekV2Config {
  "architectures": [
    "DeepseekV2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_deepseek.DeepseekV2Config",
    "AutoModel": "modeling_deepseek.DeepseekV2Model",
    "AutoModelForCausalLM": "modeling_deepseek.DeepseekV2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 100000,
  "eos_token_id": 100001,
  "ep_size": 1,
  "first_k_dense_replace": 1,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 10944,
  "kv_lora_rank": 512,
  "max_position_embeddings": 163840,
  "model_type": "deepseek_v2",
  "moe_intermediate_size": 1408,
  "moe_layer_freq": 1,
  "n_group": 1,
  "n_routed_experts": 64,
  "n_shared_experts": 2,
  "norm_topk_prob": false,
  "num_attention_heads": 16,
  "num_experts_per_tok": 6,
  "num_hidden_layers": 27,
  "num_key_value_heads": 16,
  "pretraining_tp": 1,
  "q_lora_rank": null,
  "qk_nope_head_dim": 128,
  "qk_ro

/home/p84400019/projects/consciousness-llms/IT-LLMs/src/activation_recorder/ActivationRecorder.py:351: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  gate_value = torch.tensor(topk_weights[expert_idx_in_weights])


Attention layer 13 activations captured with 16 heads.
[MoE Inference] new_x.shape=torch.Size([114, 2048]), topk_ids.shape=torch.Size([19, 6]), topk_weight.shape=torch.Size([19, 6])
[MoE Inference] out_before_mul.shape=torch.Size([19, 6, 2048]), topk_weight.shape=torch.Size([19, 6])
[MoE Inference] out_after_mul.shape=torch.Size([19, 6, 2048])
[MoE Inference] summed_out.shape=torch.Size([19, 2048])
[MoE Inference] y.shape=torch.Size([19, 2048]), topk_idx.shape=torch.Size([19, 6]), topk_weight.shape=torch.Size([19, 6])
[MoE Inference] y.shape=torch.Size([1, 19, 2048])
Hook for DeepseekV2MoE layer 13 which is_moe_layer=True, captured the following module output: 2
topk_ids shape: torch.Size([19, 6])
topk_weights shape: torch.Size([19, 6])
out_before_mul shape: torch.Size([19, 6, 2048])
out_after_mul shape: torch.Size([19, 6, 2048])
shared_experts_out shape: torch.Size([1, 19, 2048])
---
Removing prompt tokens from activations
topk_ids shape: torch.Size([6])
topk_weights shape: torch.Size

In [ ]:
step_acts = prompt_acts.steps[4]
for layer_idx, layer_acts in step_acts.layers.items():
    print(f"Layer {layer_idx} activations:")
    moe_layer_acts = step_acts.layers[2].moe
    for expert_id, expert_acts in moe_layer_acts.experts.items():
        print(f"Expert {expert_id} activations:")
        print(repr(expert_acts))
        print("Gate value:", expert_acts.gate_value)
        print("MLP output:", expert_acts.mlp_output)
        print("Expert output:", expert_acts.expert_output)

Layer 0 activations:
Expert 0 activations:
MoEExpertActivations(layer_index=2, expert_index=0, gate_value=None, mlp_output=None, expert_output=None, is_shared=False, model_info=ModelInformation(model_name=deepseek-ai/DeepSeek-V2-Lite, model_architecture=DeepseekV2ForCausalLM, num_layers=27, num_attention_heads_per_layer=16, total_num_attention_heads=432, attention_implementation=default, hidden_size=2048, head_dim=128, attention_implementation=default))
Gate value: None
MLP output: None
Expert output: None
Expert 1 activations:
MoEExpertActivations(layer_index=2, expert_index=1, gate_value=None, mlp_output=None, expert_output=None, is_shared=False, model_info=ModelInformation(model_name=deepseek-ai/DeepSeek-V2-Lite, model_architecture=DeepseekV2ForCausalLM, num_layers=27, num_attention_heads_per_layer=16, total_num_attention_heads=432, attention_implementation=default, hidden_size=2048, head_dim=128, attention_implementation=default))
Gate value: None
MLP output: None
Expert output: No